# PaceAlgo ML — Notebook 11: Phase 1 Evaluation

Konsolidiert die Phase-1-Erweiterungen:
- **Phase 1.1:** Market Structure Layer (15 SMC-Features)
- **Phase 1.2:** Session + Timing Features (12)
- **Phase 1.3:** HTF×LTF Interactions (~10)

Plus die 15 SHAP-Top-Features aus NB 08 = **~52 Features total**.

**Strenge Grundregel:** Jedes neue Feature muss zwei Tests bestehen:
1. **SHAP-Relevanz** (mean_abs_shap > threshold)
2. **PF-Impact** (ohne Feature vs mit Feature → ΔPF > 0)

**Asset-Split-Tests:**
- Combined Modell (alle FX+Gold-Symbole gemeinsam)
- FX-only Modell (EUR, USDJPY)
- Gold-only Modell (XAU)

**Output:** Empfehlung welches finale Feature-Set + Asset-Architektur in Phase 2 verfolgt wird.

## 1. Setup + Load Raw Data

In [ ]:
import sys, os
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/pace-algo'
    DATA_RAW = Path(PROJECT_ROOT) / 'data_cache' / 'raw'
    DATA_PROCESSED = Path('/content/processed')
    DRIVE_BACKUP = Path(PROJECT_ROOT) / 'data_cache' / 'processed'
    ARTIFACTS = Path(PROJECT_ROOT) / 'artifacts'
    REPORTS_DIR = ARTIFACTS / 'reports'
    os.chdir(PROJECT_ROOT)
    !rm -rf /tmp/pace-algo
    !git clone -q https://github.com/ecoNC/pace-algo.git /tmp/pace-algo
    !cp -rf /tmp/pace-algo/core/* {PROJECT_ROOT}/core/
    print('Core code updated from GitHub')
    DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
    if len(list(DATA_PROCESSED.glob('*.parquet'))) < (len(list(DRIVE_BACKUP.glob('*.parquet'))) if DRIVE_BACKUP.exists() else 0):
        !rsync -ah {DRIVE_BACKUP}/ {DATA_PROCESSED}/
else:
    PROJECT_ROOT = os.path.abspath('..')
    os.chdir(PROJECT_ROOT)
    from core.config import DATA_RAW, DATA_PROCESSED, ARTIFACTS_REPORTS
    REPORTS_DIR = ARTIFACTS_REPORTS

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, PROJECT_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('core'):
        del sys.modules[mod]

In [ ]:
!pip install -q lightgbm shap 2>&1 | tail -1

import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from core.config import (
    FX_SYMBOLS, METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS,
    TRAIN_END, VAL_END, HTF_CONTEXT_TIMEFRAMES,
)
from core.features import (
    compute_features, attach_macro, attach_htf_context,
    compute_smc_features, compute_session_features, compute_htf_interactions,
)
from core.features.engineer import atr as atr_fn
from core.train import (
    walk_forward_split, binary_label_for_long,
    trading_metrics_from_predictions,
)
from core.analysis import confidence_percentile_sweep

R_VALUE = 1.5
TRAINING_SYMBOLS = FX_SYMBOLS + METAL_SYMBOLS
print(f'Training symbols: {TRAINING_SYMBOLS}')
print(f'Dev hold-out: {DEV_HOLDOUT_SYMBOLS}')

## 2. Build Extended Feature Set (52 Features total)

Pro Symbol: lese OHLCV, berechne base + SMC + session + HTF interaction. Speichere als `<symbol>_<tf>_extended.parquet` in `/content/processed_v2/`.

In [ ]:
EXTENDED_DIR = Path('/content/processed_v2') if IS_COLAB else (DATA_PROCESSED.parent / 'processed_v2')
EXTENDED_DIR.mkdir(parents=True, exist_ok=True)

def load_ohlcv(symbol, tf):
    p = DATA_RAW / f'{symbol}_{tf}.parquet'
    return pd.read_parquet(p) if p.exists() else None

macro_path = DATA_RAW / 'macro_daily.parquet'
macro = pd.read_parquet(macro_path) if macro_path.exists() else pd.DataFrame()

ALL_SYMBOLS = TRAINING_SYMBOLS  # incl. GBPUSD which is dev-holdout but we still compute its features for eval

for symbol in tqdm(ALL_SYMBOLS, desc='Extended features'):
    # Pre-compute HTF base features (once per symbol)
    htf_feats = {}
    for htf in HTF_CONTEXT_TIMEFRAMES:
        d = load_ohlcv(symbol, htf)
        if d is None or d.empty:
            htf_feats[htf] = pd.DataFrame()
        else:
            htf_feats[htf] = compute_features(d)

    for tf in PRIMARY_TIMEFRAMES:
        out_path = EXTENDED_DIR / f'{symbol}_{tf}_extended.parquet'
        if out_path.exists():
            print(f'  skip {symbol} {tf} (cached)')
            continue
        ohlcv = load_ohlcv(symbol, tf)
        if ohlcv is None or ohlcv.empty:
            continue

        # Base features (Trend, Momentum, Vol, Structure-light, Volume, Session-cyclic)
        base = compute_features(ohlcv)

        # Add HTF + Macro (with shift-1 leakage fix already in engineer.py)
        base = attach_htf_context(base,
                                    htf_feats.get('1h', pd.DataFrame()),
                                    htf_feats.get('4h', pd.DataFrame()))
        base = attach_macro(base, macro)

        # SMC features
        atr14 = atr_fn(ohlcv['high'], ohlcv['low'], ohlcv['close'], 14).values
        ema_align = base['ema_alignment'].fillna(0).values
        smc = compute_smc_features(ohlcv, atr14, ema_align)

        # Session features
        sess = compute_session_features(ohlcv, atr14)

        # HTF × LTF interactions
        inter = compute_htf_interactions(base)

        extended = pd.concat([base, smc, sess, inter], axis=1)
        # Load corresponding R=1.5 labels
        label_path = DATA_PROCESSED / f'labels_{symbol}_{tf}_R{R_VALUE}.parquet'
        if label_path.exists():
            labels = pd.read_parquet(label_path)
            extended = extended.join(labels[['label']], how='inner')
        extended['symbol'] = symbol
        extended['timeframe'] = tf
        extended.to_parquet(out_path, compression='zstd')
        n_feat = len([c for c in extended.columns if c not in ('symbol', 'timeframe', 'label')])
        print(f'  OK  {symbol} {tf}: {len(extended):,} rows, {n_feat} features')

    htf_feats.clear()

print('\nDone. Extended feature files saved.')

## 3. Helper Functions

In [ ]:
PARAMS_PINE = {
    'objective': 'binary', 'metric': 'binary_logloss',
    'num_leaves': 7, 'max_depth': 3, 'min_data_in_leaf': 200,
    'learning_rate': 0.05, 'num_iterations': 30, 'lambda_l2': 0.5,
    'feature_fraction': 0.85, 'bagging_fraction': 0.85, 'bagging_freq': 5,
    'is_unbalance': True, 'verbose': -1, 'n_jobs': -1,
}

def load_extended(symbol, tf):
    p = EXTENDED_DIR / f'{symbol}_{tf}_extended.parquet'
    return pd.read_parquet(p) if p.exists() else None

def stack_extended(symbols, tfs, drop_holdout):
    drop = set(drop_holdout or [])
    frames = []
    for s in symbols:
        if s in drop:
            continue
        for tf in tfs:
            d = load_extended(s, tf)
            if d is None or d.empty:
                continue
            frames.append(d)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, axis=0).sort_index()

def train_eval(df_full, feature_cols, label='?'):
    """Train Pine-budget model, return test PF and top-1% PF."""
    df_c = df_full.dropna(subset=feature_cols + ['label'])
    train_df, val_df, test_df = walk_forward_split(df_c, TRAIN_END, VAL_END)
    if len(train_df) < 1000 or len(val_df) < 100 or len(test_df) < 100:
        return None
    X_tr = train_df[feature_cols]; y_tr = binary_label_for_long(train_df['label'])
    X_vl = val_df[feature_cols]; y_vl = binary_label_for_long(val_df['label'])
    X_ts = test_df[feature_cols]
    td = lgb.Dataset(X_tr, label=y_tr)
    vd = lgb.Dataset(X_vl, label=y_vl, reference=td)
    model = lgb.train(PARAMS_PINE, td, num_boost_round=30,
                       valid_sets=[vd], valid_names=['val'],
                       callbacks=[lgb.log_evaluation(period=0)])
    val_proba = model.predict(X_vl)
    test_proba = model.predict(X_ts)
    # Tier cutoffs from VAL set
    val_sorted = np.sort(val_proba)[::-1]
    cutoff_std = float(val_sorted[max(1, int(len(val_sorted) * 0.10) - 1)])
    cutoff_high = float(val_sorted[max(1, int(len(val_sorted) * 0.03) - 1)])
    cutoff_prem = float(val_sorted[max(1, int(len(val_sorted) * 0.01) - 1)])
    # Test set metrics per tier
    test_labels = test_df['label']
    win_R = R_VALUE
    metrics_by_tier = {}
    for name, cutoff in [('Standard', cutoff_std), ('High', cutoff_high), ('Premium', cutoff_prem)]:
        mask = test_proba >= cutoff
        sub = test_labels.iloc[mask.nonzero()[0]]
        wins = int((sub == 1).sum()); losses = int((sub == -1).sum()); n = len(sub)
        pf = (wins * win_R) / losses if losses > 0 else float('inf') if wins > 0 else 0.0
        wr = wins / (wins + losses) if (wins + losses) > 0 else 0.0
        metrics_by_tier[name] = {'pf': pf, 'wr': wr, 'n': n, 'cutoff': cutoff}
    return {'model': model, 'tiers': metrics_by_tier, 'n_train': len(train_df), 'n_test': len(test_df),
             'val_proba': val_proba, 'test_proba': test_proba, 'test_df': test_df, 'feature_cols': feature_cols}

## 4. Asset-Split Experiments

Wir vergleichen:
- **Baseline**: FX+Gold combined mit ursprünglichen 15 SHAP Features
- **Extended Combined**: FX+Gold mit allen ~52 Features
- **FX-only Extended**: nur EUR, USDJPY mit allen Features
- **Gold-only Extended**: nur XAU mit allen Features

Pro Setup: Premium-Tier PF, High-Tier PF, Standard-Tier PF, Trade-Counts.

In [ ]:
# Original 15 features from NB 08
BASELINE_15 = [
    'hour_sin', 'dist_to_swing_low_atr', 'htf_1h_rsi_14', 'realized_vol_20',
    'atr_percentile_100', 'atr_pct', 'dist_to_swing_high_atr', 'volume_z_score',
    'ema_20_slope_atr', 'hour_cos', 'momentum_composite', 'rvol_20',
    'adx_14', 'ema_20_dist_atr', 'htf_1h_atr_percentile_100',
]

experiments = {}

# --- Exp 0: Baseline (FX+Gold combined, 15 features) ---
df_combined = stack_extended(TRAINING_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS)
print(f'Combined dataset: {df_combined.shape}')
available_features_combined = [f for f in BASELINE_15 if f in df_combined.columns]
print(f'Baseline-15 in combined: {len(available_features_combined)}')
experiments['0_baseline_combined'] = train_eval(df_combined, available_features_combined, label='Baseline')

# --- Exp 1: Extended Combined (all features) ---
all_features_combined = [c for c in df_combined.columns if c not in ('symbol', 'timeframe', 'label')]
print(f'All features in combined: {len(all_features_combined)}')
experiments['1_extended_combined'] = train_eval(df_combined, all_features_combined, label='Extended Combined')

# --- Exp 2: FX-only Extended (EUR + USDJPY, with GBPUSD as holdout) ---
df_fx = stack_extended(FX_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS)
print(f'FX-only dataset: {df_fx.shape}')
all_features_fx = [c for c in df_fx.columns if c not in ('symbol', 'timeframe', 'label')]
experiments['2_fx_only_extended'] = train_eval(df_fx, all_features_fx, label='FX-only')

# --- Exp 3: Gold-only Extended ---
df_gold = stack_extended(METAL_SYMBOLS, PRIMARY_TIMEFRAMES, DEV_HOLDOUT_SYMBOLS)
print(f'Gold-only dataset: {df_gold.shape}')
all_features_gold = [c for c in df_gold.columns if c not in ('symbol', 'timeframe', 'label')]
experiments['3_gold_only_extended'] = train_eval(df_gold, all_features_gold, label='Gold-only')

print('\nAll 4 experiments trained.')

## 5. Comparison Table

In [ ]:
rows = []
for exp_name, res in experiments.items():
    if res is None:
        continue
    for tier_name, m in res['tiers'].items():
        rows.append({
            'experiment': exp_name,
            'tier': tier_name,
            'n_trades': m['n'],
            'profit_factor': m['pf'],
            'win_rate': m['wr'],
            'cutoff': m['cutoff'],
            'n_features': len(res['feature_cols']),
            'n_train': res['n_train'],
            'n_test': res['n_test'],
        })
summary = pd.DataFrame(rows)
print('Per-experiment × per-tier (honest VAL→TEST cutoffs):')
display(summary.round(4))

# Pivot for easier reading
pivot_pf = summary.pivot_table(index='experiment', columns='tier', values='profit_factor').round(4)
print('\nProfit Factor pivot:')
display(pivot_pf[['Standard', 'High', 'Premium']])

pivot_n = summary.pivot_table(index='experiment', columns='tier', values='n_trades')
print('\nTrade Count pivot:')
display(pivot_n[['Standard', 'High', 'Premium']])

## 6. SHAP Analysis on Extended Combined Model

Welche der neuen Features (SMC, Session, HTF-Interaction) tragen tatsächlich bei?

In [ ]:
ext = experiments.get('1_extended_combined')
if ext is None:
    print('Extended combined experiment not available — skipping SHAP')
else:
    model = ext['model']
    feature_cols = ext['feature_cols']
    test_df = ext['test_df']
    # SHAP on 10k sample
    rng = np.random.default_rng(42)
    sample_idx = rng.choice(len(test_df), size=min(10_000, len(test_df)), replace=False)
    X_sample = test_df.iloc[sample_idx][feature_cols]
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    mean_abs = np.abs(sv).mean(axis=0)
    shap_table = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': mean_abs,
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)
    print(f'SHAP ranking ({len(shap_table)} features):')
    display(shap_table.head(40))

    # Classify by feature group
    SMC_FEATS = ['sweep_up_recent', 'sweep_down_recent', 'bars_since_sweep_up', 'bars_since_sweep_down',
                  'eqhigh_present', 'eqlow_present', 'dist_to_sh_atr_conf', 'dist_to_sl_atr_conf',
                  'bos_up_strict', 'bos_down_strict', 'choch_to_down', 'choch_to_up',
                  'fvg_bull_active', 'fvg_bear_active', 'dist_to_bull_fvg_atr', 'dist_to_bear_fvg_atr',
                  'fvg_bull_size_atr', 'fvg_bear_size_atr']
    SESS_FEATS = ['in_asia', 'in_london', 'in_ny', 'in_london_ny_killzone',
                   'in_asia_london_overlap', 'in_us_open_killzone', 'in_london_open_killzone',
                   'is_fx_market_open', 'vol_expansion_ratio', 'vol_expanding', 'vol_contracting',
                   'bars_since_vol_spike']
    INTER_FEATS = ['htf_ltf_agree_bull', 'htf_ltf_agree_bear', 'htf_ltf_counter_trend',
                    'htf_ltf_alignment_score', 'ltf_rsi_minus_htf_rsi',
                    'both_rsi_oversold', 'both_rsi_overbought', 'vol_pct_diff_htf',
                    'both_high_vol', 'both_low_vol', 'pullback_in_bull', 'pullback_in_bear']

    def classify(f):
        if f in SMC_FEATS: return 'SMC'
        if f in SESS_FEATS: return 'Session/Vol'
        if f in INTER_FEATS: return 'HTF-Interaction'
        if f in BASELINE_15: return 'Baseline'
        return 'Other'
    shap_table['group'] = shap_table['feature'].apply(classify)

    # Per-group SHAP contribution
    group_summary = shap_table.groupby('group')['mean_abs_shap'].agg(['sum', 'mean', 'count']).sort_values('sum', ascending=False)
    print('\nPer-group SHAP contribution:')
    display(group_summary)

    # Dead features (essentially zero SHAP)
    dead = shap_table[shap_table['mean_abs_shap'] < 1e-5]
    print(f'\nDead features (no SHAP contribution): {len(dead)} of {len(shap_table)}')
    if len(dead) > 0 and len(dead) < 40:
        print(list(dead['feature']))

## 7. SHAP-Filtered Retrain

Behalte nur die Top-N Features die signifikant beitragen — siehe ob das den Premium-Tier-PF weiter verbessert.

In [ ]:
if 'shap_table' in dir():
    for top_n in [15, 20, 25]:
        selected = list(shap_table.head(top_n)['feature'])
        res = train_eval(df_combined, selected, label=f'SHAP-Top-{top_n}')
        if res is None:
            continue
        exp_key = f'4_shap_top{top_n}'
        experiments[exp_key] = res
        prem = res['tiers']['Premium']
        high = res['tiers']['High']
        print(f'SHAP-Top-{top_n}: Premium PF {prem["pf"]:.3f} (n={prem["n"]:,}), High PF {high["pf"]:.3f}')

# Rebuild summary with new exps
rows = []
for exp_name, res in experiments.items():
    if res is None:
        continue
    for tier_name, m in res['tiers'].items():
        rows.append({'experiment': exp_name, 'tier': tier_name,
                      'n_trades': m['n'], 'profit_factor': m['pf'], 'win_rate': m['wr']})
summary_all = pd.DataFrame(rows)
pivot_pf_all = summary_all.pivot_table(index='experiment', columns='tier', values='profit_factor').round(4)
print('\nFinal PF Comparison:')
display(pivot_pf_all[['Standard', 'High', 'Premium']])

## 8. Verdict + Recommendation

In [ ]:
print('=' * 70)
print('PHASE 1 EVALUATION SUMMARY')
print('=' * 70)

premium_pfs = {}
for exp_name, res in experiments.items():
    if res is None:
        continue
    premium_pfs[exp_name] = res['tiers']['Premium']['pf']

print('\nPremium-Tier PF per setup:')
for exp_name, pf in sorted(premium_pfs.items(), key=lambda x: -x[1]):
    print(f'  {exp_name:<30s}: {pf:.4f}')

best_name = max(premium_pfs, key=premium_pfs.get)
best_pf = premium_pfs[best_name]
baseline_pf = premium_pfs.get('0_baseline_combined', 1.79)
lift = best_pf - baseline_pf

print(f'\nBest setup: {best_name} (PF {best_pf:.3f})')
print(f'Baseline (NB 08 reference): PF {baseline_pf:.3f}')
print(f'Phase 1 lift: {lift:+.3f} PF')

if best_pf >= 1.9:
    print('\n  -> PHASE 1 TARGET MET (Premium PF >= 1.9). Consider Phase 2 or proceed to Pine.')
elif best_pf >= 1.8:
    print('\n  -> Strong improvement. Phase 2 (FVG refinement, cross-asset) likely pushes us to 2.0+.')
else:
    print('\n  -> Marginal improvement. Need to investigate why new features did not lift PF.')

---

Schick mir nach dem Run:
- Section 5 (Comparison Table)
- Section 6 (SHAP per group)
- Section 7 (SHAP-filtered retrain)
- Section 8 (Verdict)

Daraus entscheiden wir Phase 2 Priorisierung.